# Making a request

In [14]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [15]:
message = client.messages.create(
    model=model,
    max_tokens=100,
    messages=[{"role": "user", "content": "What is quantum computing? Answer in one sentence"}]
)

In [16]:
message.content[0].text

'Quantum computing is a type of computing that uses quantum bits (qubits) to perform calculations by exploiting quantum mechanical properties like superposition and entanglement, potentially solving certain complex problems much faster than classical computers.'

# Multi-Turn Conversations

In [ ]:
def _add_message(messages: list, role: str, text: str) -> None:
    message = {"role": role, "content": text}
    messages.append(message)

def add_user_message(messages: list, text: str) -> None:
    _add_message(messages, "user", text)

def add_assistant_message(messages: list, text: str) -> None:
    _add_message(messages, "assistant", text)

def chat(messages: list) -> str:
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages
    )
    return response.content[0].text

In [18]:
messages = [] # Initialize an empty list to hold the conversation history

add_user_message(messages, "What is quantum computing? Answer in one sentence")  # Add the user's message using the helper function

llm_response = chat(messages)  # Get the assistant's response based on the conversation history
print(llm_response)  # Print the assistant's response

add_assistant_message(messages, llm_response)  # Add the assistant's response to the conversation history using the helper function

add_user_message(messages, "Write another sentence")  # Add another user message to the conversation history

llm_response = chat(messages)  # Get the assistant's response based on the updated conversation history
print(llm_response)  # Print the assistant's response


Quantum computing is a type of computing that uses quantum bits (qubits) to process information in multiple states simultaneously, potentially solving certain complex problems much faster than classical computers.
Unlike classical computers that use bits as either 0 or 1, qubits can exist in a superposition of both states at once, allowing quantum computers to explore many possible solutions in parallel.


# Chat exercise

In [20]:
messages = [] # Initialize an empty list to hold the conversation history

def format_assistant_response(response: str) -> str:
    formatted_response = "---\n"
    formatted_response += f"{response}\n"
    formatted_response += "---\n"
    return formatted_response

while True:
    user_input = input("> ")  # Get user input from the console
    print(f"> {user_input}")  # Echo the user's input back to the console
    add_user_message(messages, user_input)  # Add the user's message to the conversation history

    llm_response = chat(messages)  # Get the assistant's response based on the conversation history
    formatted_response = format_assistant_response(llm_response)
    print(formatted_response)  # Print the formatted assistant's response

    add_assistant_message(messages, llm_response)  # Add the assistant's response to the conversation history

> What is 2+5?
---
2 + 5 = 7
---

> Add 3 to that
---
7 + 3 = 10
---

> 


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.4: user messages must have non-empty content'}, 'request_id': 'req_011CaXBoMx9tAKz8rh5YEYyS'}

# System prompts

In [21]:
messages = [] # Initialize an empty list to hold the conversation history

system_prompt = """You are a patient math tutor. Do not provide direct answers, but instead guide the student to a solution step by step."""

# Overwriting the chat function to allow for a system prompt to be included in the API call
def chat(messages: list, system_prompt: str | None = None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages
    }
    if system_prompt is not None:
        params["system"] = system_prompt
    response = client.messages.create(
        **params
    )
    return response.content[0].text

add_user_message(messages, "What is the limit of 1/x?")  # Add the user's message to the conversation history

chat_response = chat(messages, system_prompt=system_prompt)  # Get the assistant's response based on the conversation history and system prompt
print(chat_response)  # Print the assistant's response


# Finding the limit of 1/x

Great question! But I need to guide you through this first—**the limit of 1/x depends on where x is approaching!** That's the key insight here.

Let me ask you some questions to help you think through this:

1. **Where is x approaching?** Is it approaching 0, infinity, a specific number like 2, or something else?

2. Once you tell me that, here's the next question: **What happens to the fraction 1/x as x gets very close to your target value?**

For example:
- If x approaches 2, what value does 1/x get close to?
- If x approaches infinity, what happens to the fraction 1/x?
- If x approaches 0... well, think carefully about this one! Does the answer depend on which direction (from the left or right)?

Try picking one of these scenarios and working through it. What do you think?


# System prompts exercise

In [22]:
messages = []

system_prompt = """You are a Python programmer.  If you are asked to write code, generate parsimonious code that is easy to read and understand.  Do not include any comments in the code unless asked to."""

add_user_message(messages, "Write a Python function that checks a string for duplicate characters.")  # Add the user's message to the conversation history
chat_response = chat(messages, system_prompt=system_prompt)  # Get the assistant's response based on the conversation history and system prompt
print(chat_response)  # Print the assistant's response

```python
def has_duplicates(s):
    return len(s) != len(set(s))
```

This function converts the string to a set (which removes duplicates) and compares the lengths. If they differ, there are duplicate characters.

**Example usage:**
```python
print(has_duplicates("hello"))      # True
print(has_duplicates("abc"))        # False
print(has_duplicates("aabbcc"))     # True
```

**Alternative version** if you need to find which characters are duplicated:

```python
def get_duplicates(s):
    return set([c for c in s if s.count(c) > 1])
```

**Example:**
```python
print(get_duplicates("hello"))      # {'l'}
print(get_duplicates("aabbcc"))     # {'a', 'b', 'c'}
```


# Temperature

In [24]:
# Overwriting the chat function to allow for a temperature parameter to be included in the API call
def chat(messages: list, system_prompt: str | None = None, temperature: float=1.0) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system_prompt is not None:
        params["system"] = system_prompt
    response = client.messages.create(
        **params
    )
    return response.content[0].text

In [25]:
messages = []

add_user_message(messages, "Generate a one sentence movie idea")

chat_response = chat(messages, temperature=0.0)  # Get the assistant's response with a temperature of 0.0 for more deterministic output
print(chat_response)

# Movie Idea

A disgraced archaeologist discovers that the ancient curse she dismissed as folklore is real when her team members begin dying in impossible ways, forcing her to race against time to break the spell before it claims her life.


In [26]:
messages = []

add_user_message(messages, "Generate a one sentence movie idea")

chat_response = chat(messages, temperature=1.0)  # Get the assistant's response with a temperature of 1.0 for more creative output
print(chat_response)

# Movie Idea

A disgraced magician discovers that his stage illusions are accidentally opening portals to alternate dimensions, forcing him to team up with a skeptical physicist to seal them before something dangerous slips through.


# Response streaming

In [27]:
messages = []

add_user_message(messages, "Write a one sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=100,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_019Xyn6bUi273zyQcr1DfDeS', container=None, content=[], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=16, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='#', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' Fake Database Description\n\nThe "CloudMind Analytics" database is a fictional enterprise system that', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=Tex

In [28]:
messages = []

add_user_message(messages, "Write a one sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages
) as stream:
    for text in stream.text_stream:
        print(text, end="")
    
    final_message = stream.get_final_message()

# Fake Database Description

The "CloudMirror" database is a distributed, in-memory storage system that uses quantum-entangled data nodes to achieve sub-nanosecond query response times while automatically synchronizing across parallel dimensions.

# Structured data

In [29]:
messages = []

add_user_message(messages, "Generate a very short event bridge rule in JSON format")

print(chat(messages))

```json
{
  "Name": "MyEventRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}
```


In [31]:
# overwrite chat function to allow for stop sequences to be included in the API call
def chat(messages: list, system_prompt: str | None = None, temperature: float=1.0, stop: list | None = None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system_prompt is not None:
        params["system"] = system_prompt
    if stop is not None:
        params["stop_sequences"] = stop
    response = client.messages.create(
        **params
    )
    return response.content[0].text

messages = []

add_user_message(messages, "Generate a very short event bridge rule in JSON format")
add_assistant_message(messages, "```json")

text = chat(messages, stop=["```"])
print(text)


{
  "Name": "MySimpleRule",
  "EventBusName": "default",
  "EventPattern": {
    "source": ["aws.ec2"],
    "detail-type": ["EC2 Instance State-change Notification"],
    "detail": {
      "state": ["running"]
    }
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction",
      "Id": "1"
    }
  ]
}



In [34]:
import json
json.loads(text.strip())

{'Name': 'MySimpleRule',
 'EventBusName': 'default',
 'EventPattern': {'source': ['aws.ec2'],
  'detail-type': ['EC2 Instance State-change Notification'],
  'detail': {'state': ['running']}},
 'State': 'ENABLED',
 'Targets': [{'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction',
   'Id': '1'}]}

# Structured data exercise

In [36]:
messages = []

prompt = """
Generate three different sample AWS CLI commands.  Each should be very short.
"""

add_user_message(messages, prompt)

text = chat(messages)
print(text.strip())

# Three Sample AWS CLI Commands

1. **List all S3 buckets:**
```bash
aws s3 ls
```

2. **Describe EC2 instances:**
```bash
aws ec2 describe-instances
```

3. **Get AWS account ID:**
```bash
aws sts get-caller-identity
```


In [37]:
from IPython.display import Markdown

Markdown(text)

# Three Sample AWS CLI Commands

1. **List all S3 buckets:**
```bash
aws s3 ls
```

2. **Describe EC2 instances:**
```bash
aws ec2 describe-instances
```

3. **Get AWS account ID:**
```bash
aws sts get-caller-identity
```

In [39]:
messages = []

prompt = """
Generate three different sample AWS CLI commands.  Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "```")

text = chat(messages, stop=["```"])
print(text.strip())

bash
# 1. List all S3 buckets
aws s3 ls

# 2. Describe EC2 instances
aws ec2 describe-instances

# 3. Get AWS account ID
aws sts get-caller-identity


In [40]:
messages = []

prompt = """
Generate three different sample AWS CLI commands.  Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "```bash")

text = chat(messages, stop=["```"])
print(text.strip())

aws s3 ls


In [42]:
messages = []

prompt = """
Generate three different sample AWS CLI commands.  Each should be very short.
"""

add_user_message(messages, prompt)
add_assistant_message(messages, "Here are all three commands in a single block without any comments: \n```bash")

text = chat(messages, stop=["```"])
print(text.strip())

aws s3 ls
aws ec2 describe-instances --region us-east-1
aws dynamodb list-tables
